## Domain Adaptation Pipeline

Implements DANN (Domain Adversarial Neural Network) for cross-subject EEG motor imagery classification.
Uses EEGNet as feature extractor with optional conditional discriminator (CDAN extension).

### Architecture
- Feature extractor: EEGNet (Lawhern et al. 2018)
- Classifier: two fully connected layers
- Domain discriminator: gradient reversal layer + fully connected layers
- Optional: conditional discriminator (CDAN, based on classifier predictions)

### Research Questions
- RQ1: Which neural patterns does the model rely on during cross-subject transfer?
- RQ2: Are identified features consistent across subjects and can they explain generalisation differences?

### References
- Ganin et al. 2016: original DANN
- Lawhern et al. 2018: EEGNet
- CDAN conditional extension: based on paper by same name (MDPI, open access)

### Domain adaptation pipeline imports and config

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import Function

from pathlib import Path
import random
import math

from itertools import cycle

import copy
import json
import csv
from datetime import datetime
import os

from sklearn.metrics import cohen_kappa_score, roc_auc_score, accuracy_score
from scipy.spatial.distance import cdist
from scipy.signal import welch
from sklearn.metrics import confusion_matrix

import pyriemann
from pyriemann.utils.covariance import covariances
from pyriemann.utils.distance import distance_riemann
from pyriemann.utils.mean import mean_riemann
from pyriemann.estimation import Covariances
from pyriemann.utils.mean import mean_covariance
from pyriemann.utils.base import invsqrtm, sqrtm

# reproducibility
np.random.seed(25)
random.seed(25)
torch.manual_seed(25)

In [3]:
import sys
sys.path.insert(0, '../functions')

from models import (
    GradientReversalFunction, GradientReversalLayer,
    DomainDiscriminator, ConditionalDomainDiscriminator,
    Classifier, EEGNet, InceptionBranch, EEGSymAdapted, DANN
)
from metrics import (
    extract_band_power_features, compute_mmd_multiscale, compute_mmd_optimal_bandwidth,
    compute_riemannian_distance, apply_riemannian_alignment, compute_subject_mmd,
    compute_pre_adaptation_mmd, compute_post_adaptation_mmd, evaluate
)
from augmentation import patch_perturbation, hemisphere_perturbation, random_shift, apply_augmentation
from data_loading import (
    load_subject_data, load_subject_data_windowed, load_subject_data_multiclass,
    load_subject_data_16ch, load_subject_data_multiclass_16ch, load_subject_data_flexible,
    prepare_dataloaders
)
from training_utils import (
    get_alpha, get_optimizer, EarlyStopping, BestModelCheckpoint,
    initialise_log, apply_adabn, train_epoch
)
from pipeline import train_dann_single_fold, save_results, save_summary_csv, run_loso

In [4]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# print(f"Device: {device}")
# print(f"PyTorch version: {torch.__version__}")

#### Config

In [5]:
# paths
base_path = Path("..")
processed_path = base_path / "Data/processed"
results_path = base_path / "results"
multiclass_path = base_path / "Data/processed_multiclass"
ch16_path = base_path / "Data/processed_16ch"

# model config
use_conditional_discriminator = False # set True for CDAN extension

# dataset parameters
n_channels = 8
n_timepoints = 480
n_classes = 2

# print(f"Processed data path exists: {processed_path.exists()}")

#### Valid Subjects

In [6]:
# 106 subjects after removing S088, S092, S100 (incompatible sampling rate)
excluded = ['S088', 'S092', 'S100']

valid_subjects = []
for i in range(1, 110):
    subject_id = f'S{str(i).zfill(3)}'
    subject_file = processed_path / f'{subject_id}_processed.npz'
    if subject_file.exists() and subject_id not in excluded:
        valid_subjects.append(subject_id)

print(f"Valid subjects: {len(valid_subjects)}")
print(f"First five: {valid_subjects[:5]}")
print(f"Last five: {valid_subjects[-5:]}")

Valid subjects: 106
First five: ['S001', 'S002', 'S003', 'S004', 'S005']
Last five: ['S105', 'S106', 'S107', 'S108', 'S109']


## Functions end here

In [7]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

### Checkpoints saving run

In [ ]:
subset_60 = ['S002', 'S003', 'S005', 'S006', 'S009', 'S010', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S021', 'S007', 'S029', 'S018', 'S037','S024', 'S025', 'S026', 'S028', 'S029', 'S032', 'S033', 'S035', 'S036', 'S038', 'S040', 'S041', 'S046', 'S047', 'S049', 'S050', 'S052', 'S053', 'S054', 'S055', 'S057', 'S059', 'S061', 'S062', 'S066', 'S068', 'S071', 'S073', 'S074', 'S075', 'S076', 'S078', 'S080', 'S081', 'S082', 'S086', 'S089', 'S090', 'S094', 'S097', 'S098', 'S099', 'S101', 'S102', 'S103', 'S105', 'S106']

### Checkpoint run CDAN

In [ ]:
run_loso(
    valid_subjects=valid_subjects,
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_16ch=True, n_classes=2,
    save_checkpoint=True,
    checkpoint_tag='cdan_16ch_binary',
    results_filename='loso_results_cdan_16ch_RETRAIN.json'
)

### Check point run multiclass

##### add checkpoint_tag (checkpoint_tag='') (multiclass_16ch) when running next

In [ ]:
run_loso(
    valid_subjects=valid_subjects, stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_multiclass=True, use_16ch=True, n_classes=4,save_checkpoint=True,
    checkpoint_tag='multiclass_16ch',
    results_filename='loso_results_multiclass_16ch_with_checkpoints.json'
);
print("multiclass+16ch with_checkpoints full LOSO done")

#### test run

In [ ]:
# quick subset test CDAN vs DANN comparison
# 10 subjects, 20 epochs, kappa stopping

subset = valid_subjects[:11]

model_dann, log_dann = train_dann_single_fold(
    target_subject=subset[0],
    valid_subjects=subset,
    stopping_criterion='kappa',
    n_epochs_max=20,
    min_epochs=10,
    warmup_epochs=10,
    use_gradient_penalty=True,
    use_conditional_discriminator=False
)

model_cdan, log_cdan = train_dann_single_fold(
    target_subject=subset[0],
    valid_subjects=subset,
    stopping_criterion='kappa',
    n_epochs_max=20,
    min_epochs=10,
    warmup_epochs=10,
    use_gradient_penalty=True,
    use_conditional_discriminator=True,
    use_entropy_weighting=True
)

print(f"DANN  - Final kappa: {log_dann['final_kappa']:.4f} | Domain loss: {log_dann['domain_loss'][-1]:.4f}")
print(f"CDAN  - Final kappa: {log_cdan['final_kappa']:.4f} | Domain loss: {log_cdan['domain_loss'][-1]:.4f}")

### No Adaptation Baseline
Trains EEGNet classifier on source subjects without domain adaptation.
Lambda domain set to 0.0, GRL has no effect on feature extractor.
Provides comparison point for DANN results.
Same architecture, same data, same evaluation protocol.
Only difference: no adversarial domain alignment.


In [ ]:
# no adaptation baseline
# full LOSO run with lambda_domain=0.0
# domain loss has no effect on training
# pure classification generalisation without domain adaptation

baseline_results_file = results_path / 'loso_results_baseline.json'
baseline_summary_file = results_path / 'loso_summary_baseline.csv'

# temporarily point save functions to baseline files
# we will handle this by passing different paths

def run_loso_baseline(
    valid_subjects,
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    resume=True
):
    """
    runs LOSO with no domain adaptation
    lambda_domain=0.0 means GRL has no effect
    only kappa stopping criterion needed for baseline
    """
    if resume and baseline_results_file.exists():
        with open(baseline_results_file, 'r') as f:
            results = json.load(f)
        print(f"Resuming baseline: {len(results)} subjects already processed")
    else:
        results = {}
        print(f"Starting fresh baseline run")

    from datetime import datetime
    total_subjects = len(valid_subjects)
    start_time = datetime.now()

    print(f"Total subjects: {total_subjects}")
    print(f"Lambda domain: 0.0 (no adaptation)")
    print()

    for idx, subject_id in enumerate(valid_subjects):
        if resume and subject_id in results:
            print(f"Skipping {subject_id} (already processed)")
            continue

        subject_start = datetime.now()
        print(f"Processing {subject_id} ({idx+1}/{total_subjects})")

        try:
            model, log = train_dann_single_fold(
                target_subject=subject_id,
                valid_subjects=valid_subjects,
                stopping_criterion='kappa',
                n_epochs_max=n_epochs_max,
                batch_size=batch_size,
                optimizer_choice=optimizer_choice,
                lambda_domain=0.0,  # no domain adaptation
                min_epochs=0,
                warmup_epochs=0
            )

            results[subject_id] = {
                'final_accuracy': log['final_accuracy'],
                'final_kappa': log['final_kappa'],
                'final_auc': log['final_auc'],
                'log': log
            }

            print(f"  accuracy {log['final_accuracy']:.4f} | "
                  f"kappa {log['final_kappa']:.4f} | "
                  f"auc {log['final_auc']:.4f} | "
                  f"epochs {log['total_epochs_run']}")

        except Exception as e:
            print(f"  Error for {subject_id}: {e}")
            results[subject_id] = None

        # save after each subject
        with open(baseline_results_file, 'w') as f:
            def convert(obj):
                if isinstance(obj, (bool,)):
                    return bool(obj)
                elif hasattr(obj, 'item'):
                    return obj.item()
                return obj

            def recursive_convert(d):
                if isinstance(d, dict):
                    return {k: recursive_convert(v) for k, v in d.items()}
                elif isinstance(d, list):
                    return [recursive_convert(i) for i in d]
                else:
                    return convert(d)

            json.dump(recursive_convert(results), f, indent=2)

        subject_time = datetime.now() - subject_start
        elapsed = datetime.now() - start_time
        avg_time = elapsed / (idx + 1)
        remaining = avg_time * (total_subjects - idx - 1)

        print(f"  Time: {subject_time.seconds}s | "
              f"Elapsed: {elapsed.seconds//60}m | "
              f"Remaining: {remaining.seconds//60}m")
        print()

    print(f"Baseline complete. Results saved to {baseline_results_file}")
    return results

In [ ]:
# # verify baseline on small subset
# small_baseline = run_loso_baseline(
#     valid_subjects=['S001', 'S002', 'S003', 'S004', 'S005', 'S006'],
#     n_epochs_max=10,
#     resume=False
# )

# print(f"\nBaseline verification complete")
# print(f"Subjects processed: {len(small_baseline)}")

In [ ]:
# full baseline LOSO run
baseline_full = run_loso_baseline(
    valid_subjects=valid_subjects,
    n_epochs_max=100,
    resume=True
)

print(f"Full baseline complete")
print(f"Subjects processed: {len(baseline_full)}")

#### No Baseline 16 Chanbel

In [8]:
# 16-channel no-adaptation baseline
# same settings as the CDAN 16 ch run in cell 15, except lambda_domain = 0.0
from datetime import datetime

baseline16_file = results_path / 'loso_results_baseline_16ch.json'   # new file, never the 8-ch one
results = json.load(open(baseline16_file)) if baseline16_file.exists() else {}
print(f"Already done: {len(results)} subjects")

start = datetime.now()
for idx, subject_id in enumerate(valid_subjects):
    if subject_id in results:
        continue
    t0 = datetime.now()
    try:
        model, log = train_dann_single_fold(
            target_subject=subject_id,
            valid_subjects=valid_subjects,
            stopping_criterion='kappa',
            n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
            embed_dim=128, dropout_rate=0.25,
            lambda_domain=0.0,
            early_stopping_patience=10, early_stopping_min_delta=0.001,
            min_epochs=20, warmup_epochs=10,
            use_gradient_penalty=True, use_conditional_discriminator=True,
            use_entropy_weighting=True, feature_extractor_type='eegnet',
            use_16ch=True, n_classes=2,
        )
        results[subject_id] = {'final_accuracy': log['final_accuracy'],
                               'final_kappa': log['final_kappa'],
                               'final_auc': log['final_auc'], 'log': log}
        print(f"{idx+1}/{len(valid_subjects)} {subject_id}: kappa {log['final_kappa']:.4f} | "
              f"epochs {log['total_epochs_run']} | {(datetime.now()-t0).seconds}s")
    except Exception as e:
        print(f"{subject_id}: error {e}")
        results[subject_id] = None
    with open(baseline16_file, 'w') as f:   # saved after every subject, so it can resume
        json.dump(results, f, indent=2,
                  default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))

print(f"Done in {datetime.now() - start}")

Already done: 0 subjects
1/106 S001: kappa 0.6364 | epochs 32 | 117s
2/106 S002: kappa 0.8221 | epochs 40 | 127s
3/106 S003: kappa -0.0233 | epochs 33 | 111s
4/106 S004: kappa 0.6457 | epochs 32 | 111s
5/106 S005: kappa -0.0080 | epochs 33 | 111s
6/106 S006: kappa 0.1642 | epochs 35 | 117s
7/106 S007: kappa 1.0000 | epochs 31 | 108s
8/106 S008: kappa 0.6443 | epochs 32 | 111s
9/106 S009: kappa 0.3202 | epochs 31 | 109s
10/106 S010: kappa 0.5061 | epochs 42 | 131s
11/106 S011: kappa 0.5714 | epochs 31 | 109s
12/106 S012: kappa 0.7630 | epochs 31 | 108s
13/106 S013: kappa 0.7327 | epochs 31 | 108s
14/106 S014: kappa 0.3775 | epochs 42 | 134s
15/106 S015: kappa 0.9555 | epochs 31 | 109s
16/106 S016: kappa 0.5000 | epochs 33 | 112s
17/106 S017: kappa 0.2418 | epochs 43 | 134s
18/106 S018: kappa 0.0294 | epochs 43 | 134s
19/106 S019: kappa 0.6471 | epochs 31 | 108s
20/106 S020: kappa 0.2373 | epochs 32 | 110s
21/106 S021: kappa 0.6018 | epochs 38 | 124s
22/106 S022: kappa 0.3182 | epochs 39

### Full LOSO Run - DANN
-- GP Warmup, no source weighing

In [ ]:
# full LOSO run on all 106 valid subjects
# both stopping criteria per subject
# resume=True allows restart from interruption point

full_results = run_loso(
    valid_subjects=valid_subjects,
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    embed_dim=128,
    dropout_rate=0.25,
    lambda_domain=0.6,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    resume=True,
    results_filename='loso_results_sw_gp_warmup.json'
)

print(f"Full LOSO complete")
print(f"Subjects processed: {len(full_results)}")

#### Seeds run check for stability - DANN
-- Configuration: Gradient Panelty warmup, no source weighting.

In [ ]:
# seed 2: GP warmup, no source weighting
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

run_loso(
    valid_subjects=valid_subjects,
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    embed_dim=128,
    dropout_rate=0.25,
    lambda_domain=0.6,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    min_epochs=20,
    warmup_epochs=10,
    use_gradient_penalty=True,
    resume=True,
    results_filename='loso_results_gp_seed42.json'
)

print("Seed 2 complete, starting seed 3")

# seed 3: GP warmup, no source weighting
torch.manual_seed(7)
np.random.seed(7)
random.seed(7)

run_loso(
    valid_subjects=valid_subjects,
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    embed_dim=128,
    dropout_rate=0.25,
    lambda_domain=0.6,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    min_epochs=20,
    warmup_epochs=10,
    use_gradient_penalty=True,
    resume=True,
    results_filename='loso_results_gp_seed7.json'
)

print("Both seeds complete")

#### Gp Warm up 16 channels

In [ ]:
run_loso(
    valid_subjects=valid_subjects, stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=False,
    use_entropy_weighting=False, feature_extractor_type='eegnet',
    use_16ch=True, n_classes=2,
    results_filename='loso_results_gp_warmup_16ch.json'
);
print("GP+warmup 16ch full LOSO done")

### CDAN Full Loso Run

In [ ]:
print("CDAN Processing")
run_loso(
    valid_subjects=valid_subjects,
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    embed_dim=128,
    dropout_rate=0.25,
    lambda_domain=0.6,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    resume=True,
    min_epochs=20,
    warmup_epochs=10,
    use_gradient_penalty=True,
    use_conditional_discriminator=True,
    use_entropy_weighting=True,
    results_filename='loso_results_cdan.json'
);

#### Seeds Run - CDAN

In [ ]:
# CDAN seed runs - run after CDAN base run completes
# seed 2
print("Both CDAN seeds Started")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

run_loso(
    valid_subjects=valid_subjects,
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    embed_dim=128,
    dropout_rate=0.25,
    lambda_domain=0.6,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    resume=True,
    min_epochs=20,
    warmup_epochs=10,
    use_gradient_penalty=True,
    use_conditional_discriminator=True,
    use_entropy_weighting=True,
    results_filename='loso_results_cdan_seed42.json'
);

print("CDAN seed 42 complete, starting seed 7")

# seed 3
torch.manual_seed(7)
np.random.seed(7)
random.seed(7)

run_loso(
    valid_subjects=valid_subjects,
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    embed_dim=128,
    dropout_rate=0.25,
    lambda_domain=0.6,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    resume=True,
    min_epochs=20,
    warmup_epochs=10,
    use_gradient_penalty=True,
    use_conditional_discriminator=True,
    use_entropy_weighting=True,
    results_filename='loso_results_cdan_seed7.json'
);

print("Both CDAN seeds complete")

### EEG Sym loso run

In [ ]:
print("EEGSym loso run")
run_loso(
    valid_subjects=valid_subjects,
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    embed_dim=128,
    dropout_rate=0.25,
    lambda_domain=0.6,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    resume=True,
    min_epochs=20,
    warmup_epochs=10,
    use_gradient_penalty=True,
    use_conditional_discriminator=True,
    use_entropy_weighting=True,
    use_riemannian_alignment=False,
    use_adabn=False,
    feature_extractor_type='eegsym',
    results_filename='loso_results_eegsym.json'
);

### augmentation check on subset

In [ ]:
random.seed(25)
subset_60_aug = random.sample(valid_subjects, 60)

run_loso(
    valid_subjects=subset_60_aug,
    stopping_criteria=['kappa'],
    n_epochs_max=100,
    batch_size=32,
    optimizer_choice='adamw',
    embed_dim=128,
    dropout_rate=0.25,
    lambda_domain=0.6,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    resume=True,
    min_epochs=20,
    warmup_epochs=10,
    use_gradient_penalty=True,
    use_conditional_discriminator=True,
    use_entropy_weighting=True,
    feature_extractor_type='eegnet',
    use_augmentation=True,
    results_filename='subset60_augmentation.json'
);

In [ ]:
import random
random.seed(25)
subset_60 = ['S002', 'S003', 'S005', 'S006', 'S009', 'S010', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S021', 'S024', 'S025', 'S026', 'S028', 'S029', 'S032', 'S033', 'S035', 'S036', 'S038', 'S040', 'S041', 'S046', 'S047', 'S049', 'S050', 'S052', 'S053', 'S054', 'S055', 'S057', 'S059', 'S061', 'S062', 'S066', 'S068', 'S071', 'S073', 'S074', 'S075', 'S076', 'S078', 'S080', 'S081', 'S082', 'S086', 'S089', 'S090', 'S094', 'S097', 'S098', 'S099', 'S101', 'S102', 'S103', 'S105', 'S106']
assert len(subset_60) == 60, f"Expected 60 subjects, got {len(subset_60)}"
print(f"subset_60 locked: {len(subset_60)} subjects (copied from parallel notebook)")

#### embed_dim=256

In [ ]:
# # embed_dim=256
# run_loso(
#     valid_subjects=subset_60, stopping_criteria=['kappa'],
#     n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
#     embed_dim=256, dropout_rate=0.25, lambda_domain=0.6,
#     early_stopping_patience=10, early_stopping_min_delta=0.001,
#     resume=True, min_epochs=20, warmup_epochs=10,
#     use_gradient_penalty=True, use_conditional_discriminator=True,
#     use_entropy_weighting=True, feature_extractor_type='eegnet',
#     results_filename='subset60_embed256.json'
# );
# print("embed256 done")

#### dropout=0.15

In [ ]:
# # dropout=0.15
# run_loso(
#     valid_subjects=subset_60, stopping_criteria=['kappa'],
#     n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
#     embed_dim=128, dropout_rate=0.15, lambda_domain=0.6,
#     early_stopping_patience=10, early_stopping_min_delta=0.001,
#     resume=True, min_epochs=20, warmup_epochs=10,
#     use_gradient_penalty=True, use_conditional_discriminator=True,
#     use_entropy_weighting=True, feature_extractor_type='eegnet',
#     results_filename='subset60_dropout0.15.json'
# );
# print("dropout0.15 done")

#### dropout=0.35

In [ ]:
# # dropout=0.35
# run_loso(
#     valid_subjects=subset_60, stopping_criteria=['kappa'],
#     n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
#     embed_dim=128, dropout_rate=0.35, lambda_domain=0.6,
#     early_stopping_patience=10, early_stopping_min_delta=0.001,
#     resume=True, min_epochs=20, warmup_epochs=10,
#     use_gradient_penalty=True, use_conditional_discriminator=True,
#     use_entropy_weighting=True, feature_extractor_type='eegnet',
#     results_filename='subset60_dropout0.35.json'
# );
# print("dropout0.35 done, main notebook sweep complete")

#### Longer warmup test run on full loso

In [ ]:
# print('Longer warmup test run on full loso')
# run_loso(
#     valid_subjects=valid_subjects,
#     stopping_criteria=['kappa', 'composite'],
#     n_epochs_max=100,
#     batch_size=32,
#     optimizer_choice='adamw',
#     embed_dim=128,
#     dropout_rate=0.25,
#     lambda_domain=0.6,
#     early_stopping_patience=10,
#     early_stopping_min_delta=0.001,
#     resume=True,
#     min_epochs=30,
#     warmup_epochs=30,
#     use_gradient_penalty=True,
#     use_conditional_discriminator=True,
#     use_entropy_weighting=True,
#     use_riemannian_alignment=False,
#     use_adabn=False,
#     results_filename='loso_results_warmup30.json'
# );


### Check small changes

#### n_channel and n_timepoints

In [ ]:
model_old, _ = train_dann_single_fold(
    target_subject='S001', valid_subjects=['S001','S002','S003'],
    stopping_criterion='kappa', n_epochs_max=1, min_epochs=0, warmup_epochs=0,
    use_gradient_penalty=False, use_16ch=True
)
print("Depthwise conv kernel shape:", model_old.feature_extractor.depthwise_conv[0].weight.shape)
# kernel_size=(n_channels, 1) → dimension index 2 should be 16 if use_16ch worked, 8 if it didn't

In [ ]:
import json
with open('../notebooks/02_domain_adaptation.ipynb', 'r') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    source = ''.join(cell.get('source', []))
    if 'n_channels' in source or 'n_timepoints' in source:
        print(f"--- Cell {i} ---")
        print(source[:500])
        print()

#### Function checks

In [ ]:
# verify flexible loader matches each existing hardcoded loader exactly, same subject
test_subj = 'S001'

x1, y1 = load_subject_data_16ch(test_subj)
x2, y2 = load_subject_data_flexible(test_subj, use_16ch=True)
print("16ch match:", torch.equal(x1, x2) and torch.equal(y1, y2))

x1, y1 = load_subject_data_multiclass(test_subj)
x2, y2 = load_subject_data_flexible(test_subj, use_multiclass=True)
print("multiclass match:", torch.equal(x1, x2) and torch.equal(y1, y2))

x1, y1 = load_subject_data_multiclass_16ch(test_subj)
x2, y2 = load_subject_data_flexible(test_subj, use_multiclass=True, use_16ch=True)
print("multiclass+16ch match:", torch.equal(x1, x2) and torch.equal(y1, y2))

x1, y1 = load_subject_data(test_subj)
x2, y2 = load_subject_data_flexible(test_subj)
print("plain 8ch match:", torch.equal(x1, x2) and torch.equal(y1, y2))

x1, y1 = load_subject_data_windowed(test_subj)
x2, y2 = load_subject_data_flexible(test_subj, use_windowed_data=True)
print("windowed match:", torch.equal(x1, x2) and torch.equal(y1, y2))



In [ ]:
x1, y1 = load_subject_data('S001')
x2, y2 = load_subject_data_flexible('S001')

print("x shape:", x1.shape, "vs", x2.shape)
print("y shape:", y1.shape, "vs", y2.shape)
print("x dtype:", x1.dtype, "vs", x2.dtype)
print("x equal:", torch.equal(x1, x2))
print("y equal:", torch.equal(y1, y2))
if x1.shape == x2.shape:
    print("max abs diff in x:", (x1 - x2).abs().max().item())
if y1.shape == y2.shape:
    print("y1 unique:", torch.unique(y1))
    print("y2 unique:", torch.unique(y2))

In [ ]:
x1, y1 = load_subject_data('S001')
x2, y2 = load_subject_data_flexible('S001')
print("plain 8ch match:", torch.equal(x1, x2) and torch.equal(y1, y2))

x1, y1 = load_subject_data_windowed('S001')
x2, y2 = load_subject_data_flexible('S001', use_windowed_data=True)
print("windowed match:", torch.equal(x1, x2) and torch.equal(y1, y2))

#### domain accuracy 

In [ ]:
# verify domain accuracy logging
model_test, log_test = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=['S001', 'S002', 'S003', 'S004', 'S005'],
    stopping_criterion='kappa',
    n_epochs_max=5,
    optimizer_choice='adamw',
    min_epochs=0,
    warmup_epochs=0
)

print("Domain accuracy per epoch:")
for epoch, da in zip(log_test['epoch'], log_test['domain_accuracy']):
    print(f"  Epoch {epoch}: domain accuracy {da:.4f}")
print("Domain accuracy logging verified")

#### Domain accuracy per class no warmup

In [ ]:
model_test, log_test = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=['S001', 'S002', 'S003', 'S004', 'S005'],
    stopping_criterion='kappa',
    n_epochs_max=5,
    min_epochs=0,
    warmup_epochs=0,
    use_gradient_penalty=False
)

print("Domain accuracy breakdown per epoch:")
for i in range(len(log_test['epoch'])):
    print(f"  Epoch {log_test['epoch'][i]}: "
          f"combined {log_test['domain_accuracy'][i]:.4f} | "
          f"source {log_test['source_domain_accuracy'][i]:.4f} | "
          f"target {log_test['target_domain_accuracy'][i]:.4f}")

#### Domain accuracy per class with warmup

In [ ]:
model_warmup, log_warmup = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=['S001', 'S002', 'S003', 'S004', 'S005'],
    stopping_criterion='kappa',
    n_epochs_max=15,
    min_epochs=10,
    warmup_epochs=10,
    use_gradient_penalty=False
)

print("Domain accuracy breakdown with warmup:")
for i in range(len(log_warmup['epoch'])):
    print(f"  Epoch {log_warmup['epoch'][i]}: "
          f"combined {log_warmup['domain_accuracy'][i]:.4f} | "
          f"source {log_warmup['source_domain_accuracy'][i]:.4f} | "
          f"target {log_warmup['target_domain_accuracy'][i]:.4f}")

In [ ]:
# # verify CDAN training end to end
# model_cdan_test, log_cdan_test = train_dann_single_fold(
#     target_subject='S001',
#     valid_subjects=['S001', 'S002', 'S003', 'S004', 'S005'],
#     stopping_criterion='kappa',
#     n_epochs_max=5,
#     min_epochs=0,
#     warmup_epochs=0,
#     use_gradient_penalty=True,
#     use_conditional_discriminator=True,
#     use_entropy_weighting=True
# )

# print(f"Final kappa: {log_cdan_test['final_kappa']:.4f}")
# print(f"Domain loss epoch 1: {log_cdan_test['domain_loss'][0]:.4f}")
# print(f"Domain accuracy epoch 1: {log_cdan_test['domain_accuracy'][0]:.4f}")
# print("CDAN end to end verified")

#### filename test

In [ ]:
test_results = run_loso(
    valid_subjects=['S001', 'S002'],
    stopping_criteria=['kappa', 'composite'],
    n_epochs_max=3,
    min_epochs=0,
    warmup_epochs=0,
    use_gradient_penalty=False,
    resume=False,
    results_filename='loso_test_filename.json'
)

print(f"JSON saved: {(results_path / 'loso_test_filename.json').exists()}")
print(f"CSV saved: {(results_path / 'loso_test_filename.csv').exists()}")

os.remove(results_path / 'loso_test_filename.json')
os.remove(results_path / 'loso_test_filename.csv')

print("Verified and cleaned up")

print("Test files removed, filename fix confirmed")

#### Toggleable gradient panelty

In [ ]:
model_gp_off, log_gp_off = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=['S001', 'S002', 'S003', 'S004', 'S005'],
    stopping_criterion='kappa',
    n_epochs_max=5,
    min_epochs=0,
    warmup_epochs=0,
    use_gradient_penalty=False
)
print(f"GP off - Final kappa: {log_gp_off['final_kappa']:.4f}")

model_gp_on, log_gp_on = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=['S001', 'S002', 'S003', 'S004', 'S005'],
    stopping_criterion='kappa',
    n_epochs_max=5,
    min_epochs=0,
    warmup_epochs=0,
    use_gradient_penalty=True
)
print(f"GP on  - Final kappa: {log_gp_on['final_kappa']:.4f}")
print("Gradient penalty toggle verified")

### End of check changes 

### Multi class check

In [ ]:
# n_classes = 4

# _, log_test = train_dann_single_fold(
#     target_subject='S001',
#     valid_subjects=valid_subjects[:5],
#     stopping_criterion='kappa',
#     n_epochs_max=10,
#     min_epochs=0,
#     warmup_epochs=0,
#     use_gradient_penalty=False,
#     use_multiclass=True
# )
# print(f"Multiclass test kappa: {log_test['final_kappa']:.4f}")

# n_classes = 2  # reset

In [ ]:
_, log_cdan_test = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=valid_subjects[:5],
    stopping_criterion='kappa',
    n_epochs_max=15,
    min_epochs=10,
    warmup_epochs=5,
    use_gradient_penalty=True,
    use_conditional_discriminator=True,
    use_entropy_weighting=True,
    use_multiclass=True,
    n_classes=4
)
print(f"CDAN + 4-class test kappa: {log_cdan_test['final_kappa']:.4f}")

In [ ]:
run_loso(
    valid_subjects=subset_60, stopping_criteria=['kappa'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_multiclass=True, n_classes=4,
    results_filename='subset60_multiclass.json'
);
print("multiclass subset test done")

In [ ]:
# main notebook
run_loso(
    valid_subjects=valid_subjects, stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_multiclass=True, n_classes=4,
    results_filename='loso_results_multiclass.json'
);
print("multiclass full LOSO done")

### 16 Channel

#### Full run 16 channel

In [ ]:
# full 106-subject 16ch, both criteria
n_channels = 16

run_loso(
    valid_subjects=valid_subjects, stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_16ch=True,
    results_filename='loso_results_16ch.json'
);
print("16ch full LOSO done")

n_channels = 8

#### 16 channel seed 42

In [ ]:
n_channels = 16
torch.manual_seed(42)

run_loso(
    valid_subjects=valid_subjects, stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_16ch=True,
    results_filename='TEMP_loso_results_16ch_seed42.json'
);
print("16ch seed42 done")

n_channels = 8

### check of merge notebook

In [ ]:
_, log_cdan_test = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=valid_subjects[:5],
    stopping_criterion='kappa',
    n_epochs_max=15,
    min_epochs=10,
    warmup_epochs=5,
    use_gradient_penalty=True,
    use_conditional_discriminator=True,
    use_entropy_weighting=True,
    use_multiclass=True,
    n_classes=4
)
print(f"CDAN + 4-class test kappa: {log_cdan_test['final_kappa']:.4f}")

In [ ]:
import random
random.seed(25)
# subset_60 = ['S002','S003', 'S005', 'S006', 'S009', 'S010', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S021', 'S024', 'S025', 'S026', 'S028', 'S029', 'S032', 'S033', 'S035', 'S036', 'S038', 'S040', 'S041', 'S046', 'S047', 'S049', 'S050', 'S052', 'S053', 'S054', 'S055', 'S057', 'S059', 'S061', 'S062', 'S066', 'S068', 'S071', 'S073', 'S074', 'S075', 'S076', 'S078', 'S080', 'S081', 'S082', 'S086', 'S089', 'S090', 'S094', 'S097', 'S098', 'S099', 'S101', 'S102', 'S103', 'S105', 'S106']
# assert len(subset_60) == 60, f"Expected 60 subjects, got {len(subset_60)}"
# print(f"subset_60 locked: {len(subset_60)} subjects (copied from parallel notebook)")
subset_60 = ['S001', 'S002', 'S003', 'S004', 'S005']

In [ ]:
run_loso(
    valid_subjects=subset_60, stopping_criteria=['kappa'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_multiclass=True, n_classes=4,
    results_filename='TEMP_subset60_multiclass.json'
);
print("multiclass subset test done")

In [ ]:
subset_60 = ['S001', 'S002', 'S003', 'S004', 'S005']

n_channels = 16
run_loso(
    valid_subjects=subset_60, stopping_criteria=['kappa'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_16ch=True,
    results_filename='TEMP_subset60_16ch.json'
);
print("16ch subset test done")
n_channels = 8

### Smoke test combined (ch16 + multiclass)

In [ ]:
# Smoke Test
n_channels = 16

_, log_combined_test = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=['S001', 'S002', 'S003', 'S004', 'S005'],
    stopping_criterion='kappa',
    n_epochs_max=10,
    min_epochs=0,
    warmup_epochs=0,
    use_gradient_penalty=False,
    use_multiclass=True,
    use_16ch=True,
    n_classes=4
)
print(f"Combined multiclass+16ch test kappa: {log_combined_test['final_kappa']:.4f}")
print(f"Per-class accuracy: {log_combined_test['final_per_class_accuracy']}")

n_channels = 8

In [ ]:
_, log_verify = train_dann_single_fold(
    target_subject='S001',
    valid_subjects=['S001','S002','S003','S004','S005'],
    stopping_criterion='kappa',
    n_epochs_max=10,
    min_epochs=0,
    warmup_epochs=0,
    use_gradient_penalty=False,
    use_multiclass=True,
    use_16ch=True,
    n_classes=4
)
print(f"Kappa: {log_verify['final_kappa']:.4f}")
print(f"Per-class accuracy: {log_verify['final_per_class_accuracy']}")

In [ ]:
_, log_plain_recheck = train_dann_single_fold(
    target_subject='S001', valid_subjects=['S001','S002','S003','S004','S005'],
    stopping_criterion='kappa', n_epochs_max=15, min_epochs=10, warmup_epochs=5,
    use_gradient_penalty=False
)
print(f"Plain 8ch kappa (fuller test): {log_plain_recheck['final_kappa']:.4f}")

In [ ]:
source_loader_check, target_loader_check, target_eval_loader_check, y_target_check = prepare_dataloaders(
    'S001', ['S001','S002','S003'], batch_size=32
)
for x, y in source_loader_check:
    print("source labels seen:", torch.unique(y))
    break
for x, y in target_eval_loader_check:
    print("target eval labels seen:", torch.unique(y))
    break
print("y_target_check unique:", torch.unique(y_target_check))

#### Full run multi calss and ch16

In [ ]:
n_channels = 16

run_loso(
    valid_subjects=valid_subjects, stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_multiclass=True, use_16ch=True, n_classes=4,
    results_filename='loso_results_multiclass_16ch.json'
);
print("multiclass+16ch full LOSO done")

n_channels = 8

#### Seed 42 for multiclass channel 16

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
n_channels = 16

run_loso(
    valid_subjects=valid_subjects, stopping_criteria=['kappa', 'composite'],
    n_epochs_max=100, batch_size=32, optimizer_choice='adamw',
    embed_dim=128, dropout_rate=0.25, lambda_domain=0.6,
    early_stopping_patience=10, early_stopping_min_delta=0.001,
    resume=True, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True, feature_extractor_type='eegnet',
    use_multiclass=True, use_16ch=True, n_classes=4,
    results_filename='loso_results_multiclass_16ch_seed42.json'
);
print("multiclass+16ch seed 42 full LOSO done")

n_channels = 8

##### Data leakage check for multiclass + Channel 16

In [ ]:
import numpy as np
path = Path("../Data/processed_multiclass_16ch")
d1 = np.load(path / "S003_multiclass_16ch.npz")
d2 = np.load(path / "S010_multiclass_16ch.npz")
print(f"Identical data arrays: {np.array_equal(d1['data'], d2['data'])}")  # MUST be False
print(f"S003 shape: {d1['data'].shape} | S010 shape: {d2['data'].shape}")
print(f"S003 first 5 labels: {d1['labels'][:5]} | S010 first 5: {d2['labels'][:5]}")

In [ ]:
# direct leakage check on the actual dataloader path used in training
source_loader, target_loader, target_eval_loader, y_target = prepare_dataloaders(
    'S010', valid_subjects, batch_size=32, use_multiclass=True, use_16ch=True
)
x_target_eval = next(iter(target_eval_loader))[0]
x_source_all = next(iter(source_loader))[0]
print(f"Target eval shape: {x_target_eval.shape} | Source batch shape: {x_source_all.shape}")
# spot check: is any target trial's data identical to any source trial in this batch?
for i in range(min(5, x_target_eval.shape[0])):
    matches = [torch.equal(x_target_eval[i], x_source_all[j]) for j in range(x_source_all.shape[0])]
    print(f"Target trial {i} found identical in source batch: {any(matches)}")

In [ ]:
import numpy as np
from pathlib import Path

path = Path("../Data/processed_multiclass_16ch")
all_data = {}
for subj in valid_subjects:
    f = path / f"{subj}_multiclass_16ch.npz"
    if f.exists():
        all_data[subj] = np.load(f)['data']

# check S010's first trial against every OTHER subject's every trial
target_trial = all_data['S010'][0]
duplicates_found = []
for subj, data in all_data.items():
    if subj == 'S010':
        continue
    for i in range(data.shape[0]):
        if np.array_equal(data[i], target_trial):
            duplicates_found.append((subj, i))

print(f"S010 trial 0 found duplicated in: {duplicates_found}")

In [ ]:
source_loader, target_loader, target_eval_loader, y_target = prepare_dataloaders(
    'S010', valid_subjects, batch_size=32, use_multiclass=True, use_16ch=True
)
x_target_eval = next(iter(target_eval_loader))[0]
x_source_all = next(iter(source_loader))[0]
matches = [torch.equal(x_target_eval[0], x_source_all[j]) for j in range(x_source_all.shape[0])]
print(f"Target trial 0 found identical in source batch: {any(matches)}")

In [ ]:
import inspect
print(inspect.getsource(prepare_dataloaders))

#### end of checks for multi class and channel 16

## refactor implement

### base numbers in backup fresh restarted kernel

### Post refactor checks

In [ ]:
torch.manual_seed(25)
np.random.seed(25)
random.seed(25)

model_verify, log_verify = train_dann_single_fold(
    target_subject='S001', valid_subjects=valid_subjects,
    stopping_criterion='kappa', n_epochs_max=100, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True
)
print(f"Refactored pipeline S001 kappa: {log_verify['final_kappa']:.4f}")

Refactored pipeline S001 kappa: 0.6818


In [ ]:
torch.manual_seed(25)
np.random.seed(25)
random.seed(25)

model_verify2, log_verify2 = train_dann_single_fold(
    target_subject='S001', valid_subjects=valid_subjects,
    stopping_criterion='kappa', n_epochs_max=100, min_epochs=20, warmup_epochs=10,
    use_gradient_penalty=True, use_conditional_discriminator=True,
    use_entropy_weighting=True
)
print(f"Refactored pipeline S001 kappa, second run: {log_verify2['final_kappa']:.4f}")

Refactored pipeline S001 kappa, second run: 0.6818
